# Week 10, Day 2 Lab — Real APIs, Error Handling & Retries
### From fake data to the live internet, in 2.5 hours

**Yesterday** your agent called fake, hardcoded functions. **Today every tool hits a real,
live service** — a real weather API, a real Wikipedia search, and a real (local) database.
Real services fail in ways fake ones never do, so today is also about making your agent
survive that.

**How to use this notebook:**
- **WRITE THIS** cells have only a spec/signature — you write the body.
- **TODO** cells ask you to modify or extend something already there.
- Plain cells are given — run them as-is.
- Stuck 5+ minutes on a WRITE THIS cell? The **Appendix** at the very bottom has solutions.

**Total time: 150 minutes**, including one 10-minute break.


---
## Section 0 — Setup

Same as yesterday: Gemini 2.5 Flash via Google's free `google-genai` SDK.

If you don't already have a key from Day 1: go to **https://aistudio.google.com/apikey**,
sign in, click **Create API key**, copy it. (If your Colab runtime restarted, you'll need to
re-enter it even if you used one yesterday — nothing is saved between sessions.)


In [62]:
!pip install -q google-genai


In [63]:
import os
import json
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


Enter your Gemini API key: ··········


In [69]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"

test = client.models.generate_content(model=MODEL, contents="Say 'Day 2 lab is ready!'")
print(test.text)

Day 2 lab is ready!


---
## Section 1 — Recap: Fake vs. Real (15 minutes)

Yesterday's `get_weather` looked like this:

```python
fake_data = {"lahore": {"forecast": "sunny", "temp_c": 34}, ...}
```

Today, let's hit an **actual live weather API** directly — no agent, no tool schema yet,
just a plain HTTP request — so you can see what "real" costs us that "fake" didn't.


In [70]:
import requests

# Open-Meteo is a free, real weather API that needs NO API key at all.
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 31.5497, "longitude": 74.3436, "current_weather": True},
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])


{'time': '2026-09-02T19:30', 'interval': 900, 'temperature': 29.7, 'windspeed': 6.6, 'winddirection': 123, 'is_day': 0, 'weathercode': 0}


In [71]:
# 🔧 TODO: change these coordinates to a city of your choice, then re-run
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 0.0, "longitude": 0.0, "current_weather": True},  # <-- change
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])


{'time': '2026-09-02T19:30', 'interval': 900, 'temperature': 24.8, 'windspeed': 14.9, 'winddirection': 172, 'is_day': 0, 'weathercode': 1}


---
## Section 2 — Worked Example: A Real Weather Tool, End to End

Here's the answer to "what about any city name?" — Open-Meteo also has a free **geocoding**
API (also no key needed) that turns a city name into coordinates. Our real tool will chain
**two** live API calls together: geocode the city, then fetch its weather.

This is fully worked for you — read it carefully, since Sections 4–5 ask you to write tools
with this exact same shape yourself.


In [72]:
# GIVEN — the real tool function: two chained live API calls
def get_weather(city: str) -> dict:
    """Look up real, current weather for any city name using free Open-Meteo APIs."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=5
    )
    geo.raise_for_status()
    results = geo.json().get("results")
    if not results:
        return {"error": f"Could not find a location matching '{city}'"}

    lat, lon = results[0]["latitude"], results[0]["longitude"]
    resolved_name = results[0]["name"]

    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=5
    )
    weather.raise_for_status()
    current = weather.json()["current_weather"]
    return {"city": resolved_name, "temp_c": current["temperature"], "windspeed": current["windspeed"]}


# Quick manual tests — try a real city and a nonsense one
print(get_weather("Lahore"))
print(get_weather("Xyzzyxutopia"))   # should return the {"error": ...} branch, not crash


{'city': 'Lahore', 'temp_c': 29.8, 'windspeed': 7.8}
{'error': "Could not find a location matching 'Xyzzyxutopia'"}


In [73]:
# GIVEN — the schema
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get real, current weather for any city name in the world.",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "Any city name, e.g. 'Lahore' or 'Boston'"}},
        "required": ["city"]
    }
)

weather_tool = types.Tool(function_declarations=[get_weather_declaration])
config = types.GenerateContentConfig(tools=[weather_tool])
tool_registry = {"get_weather": get_weather}


### Bring back your `run_agent` loop from Day 1

Paste in the `run_agent` function you built yesterday (or use the version below if you'd
rather start from a clean copy — it's identical to the Day 1 Appendix solution).


In [74]:
# GIVEN — carried over from Day 1 (Appendix solution, in case you don't have your own copy handy)
def run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)
            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")
            result = tool_registry[name](**args)
            if verbose:
                print(f"[Step {step+1}] Observation: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."


In [75]:
answer = run_agent(
    "What's the weather in TokoyoooooXyzabc?",
    config=config,
    tool_registry=tool_registry
)

print(answer)

[Step 1] Action: get_weather({'city': 'TokoyoooooXyzabc'})
[Step 1] Observation: {'error': "Could not find a location matching 'TokoyoooooXyzabc'"}
[Step 2] Thought: I have enough info. Giving final answer.
It looks like that city name isn't recognized. Did you mean **Tokyo**? Let me know and I can check the weather for you!


**Notice:** this used yesterday's exact loop, unmodified — only the tool underneath changed
from fake to real. That's the whole point of building a generic loop: the tools are swappable.

**🔧 TODO Checkpoint (5 min):** ask about a city you're confident doesn't exist (misspell one
badly) and confirm your agent handles the `{"error": ...}` gracefully in its final answer,
rather than crashing.


In [76]:
import time

try:
    answer = run_agent(
        "What's the weather in Tokoyooblahhhhh?",
        config=config,
        tool_registry=tool_registry
    )
    print(answer)
except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR MSG:", str(e))

[Step 1] Action: get_weather({'city': 'Tokyo'})
[Step 1] Observation: {'city': 'Tokyo', 'temp_c': 24.3, 'windspeed': 3.0}
[Step 2] Thought: I have enough info. Giving final answer.
It looks like you meant **Tokyo**! The current weather there is **24.3°C** with a wind speed of **3 km/h**.


---
## Section 3 — Error Handling & Retries (25 minutes)

Real APIs time out, rate-limit you, or occasionally the model passes an argument that doesn't
match what your function expects. Right now, if `get_weather` throws an uncaught exception
(e.g. a network timeout), your entire `run_agent` call crashes. Let's fix that at two levels.

### Level 1 — Safe argument handling

If the model ever calls a tool with the wrong argument names/types, `tool_registry[name](**args)`
raises a `TypeError` that currently kills the whole loop. Wrap it.


In [77]:
def safe_call_tool(name, args, tool_registry):
    try:
        return tool_registry[name](**args)
    except TypeError as e:
        return {"error": f"invalid arguments for {name}: {e}"}
    except Exception as e:
        return {"error": f"unexpected error in {name}: {e}"}

In [ ]:
# WRITE THIS: safe_call_tool(name, args, tool_registry) -> dict
#
# Spec:
# - Try calling tool_registry[name](**args) and return its result
# - If it raises a TypeError (wrong/missing arguments), return {"error": f"Invalid arguments for {name}: {e}"}
# - If it raises ANY other Exception, return {"error": f"Unexpected error in {name}: {e}"}
# - Never let an exception escape this function

def safe_call_tool(name, args, tool_registry):
    pass


# Test: this should print an error dict, NOT raise an exception
print(safe_call_tool("get_weather", {"wrong_argument_name": "Lahore"}, tool_registry))


### Level 2 — Retry with exponential backoff

Network calls fail transiently sometimes — a retry a second later often just works. Write the
retry wrapper from the spec below


In [ ]:
# WRITE THIS: call_with_retry(func, max_retries=3, **kwargs) -> result or error dict
#
# Spec:
# - Try calling func(**kwargs) and return its result if it succeeds
# - If it raises requests.exceptions.RequestException:
#     - if this was the last allowed attempt, return {"error": f"Failed after {max_retries} attempts: {e}"}
#     - otherwise, sleep for (2 ** attempt) seconds, then try again  (1s, 2s, 4s, ...)
# - Import `time` yourself

def call_with_retry(func, max_retries=3, **kwargs):
    pass


# Test with a deliberately broken URL-based function to confirm retries + eventual failure message
def flaky_call():
    return requests.get("https://thisdomaindefinitelydoesnotexist12345.com", timeout=2).json()

print(call_with_retry(flaky_call, max_retries=2))


In [78]:
import time
def call_with_retry(func,max_retries=3,**kwargs):
  for attempt in range(max_retries):
    try:
      return func(**kwargs)
    except requests.exceptions.RequestException as e:
      if attempt == max_retries-1:
        return {"error":f"Failed after {max_retries} attempts: {e}"}
      else:
        time.sleep(2**attempt)
def flaky_call():
    return requests.get("https://thisdomaindefinitelydoesnotexist12345.com", timeout=2).json()

print(call_with_retry(flaky_call, max_retries=2))


{'error': 'Failed after 2 attempts: HTTPSConnectionPool(host=\'thisdomaindefinitelydoesnotexist12345.com\', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d6aabacfb10>: Failed to resolve \'thisdomaindefinitelydoesnotexist12345.com\' ([Errno -2] Name or service not known)"))'}


### Harden `run_agent` with both

**🔧 TODO:** modify the loop below (a copy of your `run_agent`) so that instead of calling
`tool_registry[name](**args)` directly, it uses your two new functions together — retry the
safe call. One line changes; the rest of the loop stays the same.


In [79]:
for i, tool in enumerate(config.tools):
    for j, fn in enumerate(tool.function_declarations):
        print(f"[{i}][{j}] name={fn.name!r}")

[0][0] name='get_weather'


In [80]:
for tool in config.tools:
    tool.function_declarations = [
        fn for fn in tool.function_declarations
        if fn.name  # sirf wahi rakho jinka naam maujood hai (None/empty wale hata do)
    ]

# confirm karo ab kya bacha
for tool in config.tools:
    for fn in tool.function_declarations:
        print(fn.name)

get_weather


In [81]:
import time
import re

def call_with_retry(func, max_retries=3, **kwargs):
    last_err = None
    for attempt in range(max_retries):
        try:
            return func(**kwargs)
        except Exception as e:
            last_err = e
            msg = str(e)
            print(f"[retry {attempt+1}/{max_retries}] error: {e}")

            # agar rate-limit (429) hai to suggested delay tak wait karo
            if "RESOURCE_EXHAUSTED" in msg or "429" in msg:
                match = re.search(r"retry in (\d+(\.\d+)?)s", msg)
                wait_time = float(match.group(1)) + 2 if match else 30
                print(f"Rate limited — waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
            else:
                time.sleep(2)  # normal errors ke liye chhota wait
    raise last_err

In [82]:
def call_with_retry(func, max_retries=3, **kwargs):
    last_err = None
    for attempt in range(max_retries):
        try:
            return func(**kwargs)
        except Exception as e:
            last_err = e
            print(f"[retry {attempt+1}/{max_retries}] error: {e}")
    raise last_err


def run_agent_v2(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)

            if verbose:
                print(f"[Step {step+1}] action: {name}({args})")

            result = call_with_retry(
                safe_call_tool,
                max_retries=3,
                name=name,
                args=args,
                tool_registry=tool_registry
            )

            if verbose:
                print(f"[Step {step+1}] Observation: {result}")

            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    if verbose:
        print("[Warning] Max steps reached without final answer.")
    return response.text if response.text else "Max steps reached without a final answer."

In [83]:
import time
time.sleep(55)  # quota reset hone ka wait

answer = run_agent_v2("What's the weather in Lahore?", config=config, tool_registry=tool_registry)
print(answer)

[Step 1] action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 29.8, 'windspeed': 7.8}
[Step 2] Thought: I have enough info. Giving final answer.
The weather in Lahore is currently 29.8°C with a wind speed of 7.8 km/h.


In [84]:
# Test your hardened loop — should behave identically to run_agent for normal queries
answer = run_agent_v2("What's the weather in Lahore?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 29.8, 'windspeed': 7.8}
[Step 2] Thought: I have enough info. Giving final answer.
The current weather in Lahore is 29.8°C with a wind speed of 7.8 km/h.


---
## Section 4 — Write a Real Search Tool: Wikipedia

Your turn to build a real-API tool completely from a spec, following the exact shape of
`get_weather` in Section 2. We'll use Wikipedia's public search API — real, free, no key.

**API reference (given):**
```
GET https://en.wikipedia.org/w/api.php
params: {"action": "query", "list": "search", "srsearch": <your query>,
         "format": "json", "srlimit": <num_results>}
```
A successful response looks like:
```json
{"query": {"search": [
    {"title": "Lahore", "snippet": "Lahore is the capital of..."},
    {"title": "Lahore Fort", "snippet": "..."}
]}}
```

**Spec for the function:**
- Signature: `search_wikipedia(query: str, num_results: int = 3) -> dict`
- Call the API above with `timeout=5` and `resp.raise_for_status()`
- Return `{"results": [{"title": ..., "snippet": ...}, ...]}` using the top `num_results` hits
- If there are zero results, return `{"results": []}` (not an error — an empty result is valid)


In [85]:
import requests

def search_wikipedia(query: str, num_results: int = 3) -> dict:
    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "Week10-Day2-Lab/1.0 (educational project)"
    }

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": num_results
    }

    resp = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=5
    )

    resp.raise_for_status()

    data = resp.json()

    results = data.get("query", {}).get("search", [])

    return {
        "results": [
            {
                "title": item["title"],
                "snippet": item["snippet"]
            }
            for item in results[:num_results]
        ]
    }


print(search_wikipedia("Lahore Fort"))

{'results': [{'title': 'Lahore Fort', 'snippet': 'The <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> (Punjabi: شاہی قلعہ, romanised:\xa0Śā&#039;ī Qilā; Urdu: شاہی قلعہ, romanised:\xa0Shahi Qilah; lit.\u2009&#039;Royal <span class="searchmatch">Fort</span>&#039;) is a citadel in the walled interior'}, {'title': 'Sheesh Mahal (Lahore Fort)', 'snippet': 'located within the Shah Burj block at the north-western corner of the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span>, in <span class="searchmatch">Lahore</span>, Pakistan. It was constructed during the reign of Mughal Emperor'}, {'title': 'Lahore', 'snippet': 'shrines. <span class="searchmatch">Lahore</span> is also home to the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> and Shalimar Gardens, both of which are UNESCO World Heritage Sites. The origin of <span class="searchmatch">Lahore&#039;s</span> name is unclear'}]}


Now write the schema (same pattern as `get_weather_declaration`), register it, and test the
agent end to end with a query that needs Wikipedia.


In [86]:
search_wikipedia_declaration = types.FunctionDeclaration(
    name="search_wikipedia",
    description="Search Wikipedia for information about a topic.",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "query": types.Schema(
                type="STRING",
                description="The topic or search query to search on Wikipedia."
            ),
            "num_results": types.Schema(
                type="INTEGER",
                description="Number of Wikipedia results to return."
            )
        },
        required=["query"]
    )
)

In [87]:
# WRITE THIS: build a Tool + config with BOTH get_weather and search_wikipedia,
# and a tool_registry with both functions registered
multi_tool = types.Tool(function_declarations=[
    get_weather_declaration,
    search_wikipedia_declaration
])

config = types.GenerateContentConfig(tools=[multi_tool])

tool_registry = {
    "get_weather": get_weather,
    "search_wikipedia": search_wikipedia
}

In [88]:
for i, tool in enumerate(config.tools):
    for j, fn in enumerate(tool.function_declarations):
        print(f"[{i}][{j}] name={fn.name!r}")

[0][0] name='get_weather'
[0][1] name='search_wikipedia'


In [89]:
for tool in config.tools:
    tool.function_declarations = [
        fn for fn in tool.function_declarations
        if fn.name
    ]

for tool in config.tools:
    for fn in tool.function_declarations:
        print(fn.name)

get_weather
search_wikipedia


In [90]:
for i, tool in enumerate(config.tools):
    for j, fn in enumerate(tool.function_declarations):
        print(f"[{i}][{j}] {fn.name}")

[0][0] get_weather
[0][1] search_wikipedia


In [91]:
answer = run_agent_v2(
    "Tell me one interesting fact about Lahore from Wikipedia.",
    config=config,
    tool_registry=tool_registry
)

print(answer)

[Step 1] action: search_wikipedia({'query': 'Lahore interesting facts history architecture', 'num_results': 3})
[Step 1] Observation: {'results': [{'title': 'Culture of Lahore', 'snippet': 'Tomb of Anarkali, The <span class="searchmatch">Lahore</span> Museum, Chauburji or Char Minar and Bagh-e-Jinnah are some the major <span class="searchmatch">architectural</span> works found in <span class="searchmatch">Lahore</span>. The palaces/havelis'}, {'title': 'Pakistan', 'snippet': 'Hindustani art, seen in <span class="searchmatch">Lahore&#039;s</span> <span class="searchmatch">architectural</span> gems like the Badshahi Mosque and the <span class="searchmatch">Lahore</span> Fort with the iconic Alamgiri Gate. <span class="searchmatch">Lahore</span> also boasts the vibrant'}, {'title': 'History of Punjab', 'snippet': 'established in 1526 AD, has left an immense cultural and <span class="searchmatch">architectural</span> legacy in Punjab. The city of <span class="searchmatch">Lahore</span> bec

**🔧 Checkpoint (3 min):** ask a question that should trigger a zero-result Wikipedia search
(a nonsense query) and confirm your agent handles an empty `results` list without crashing.


In [92]:
# 🔧 write a query likely to return zero Wikipedia results, then run it
answer = run_agent_v2(
    "Tell me one interesting fact about Lahore from Wikipedia.",
    config=config,
    tool_registry=tool_registry
)

print(answer)


[Step 1] action: search_wikipedia({'query': 'Lahore history interesting facts', 'num_results': 1})
[Step 1] Observation: {'results': [{'title': 'Culture of Lahore', 'snippet': 'the capital of the Punjab province of Pakistan. <span class="searchmatch">Lahore</span> has played an important role in Pakistani <span class="searchmatch">history</span>. It was in this city that Pakistan&#039;s independence'}]}
[Step 2] action: search_wikipedia({'num_results': 1, 'query': 'Lahore facts'})
[Step 2] Observation: {'results': [{'title': 'Lahore', 'snippet': '<span class="searchmatch">Lahore</span> is the capital and largest city of the Pakistani province of Punjab. It is the second-largest city in Pakistan, after Karachi, and 27th largest'}]}
[Step 3] action: search_wikipedia({'query': 'Lahore Shalimar Gardens history', 'num_results': 1})
[Step 3] Observation: {'results': [{'title': 'Shalamar Gardens, Lahore', 'snippet': 'Shalamar <span class="searchmatch">Gardens</span> (Punjabi: شالمار باغ, roma

---
## Section 5 — Write a Real Database Tool + Test Full Chaining

Real production agents often query a real database. We'll create a small local SQLite
database (given — creating sample data isn't the learning goal here) and you'll write the
**query tool** yourself.


In [93]:
# GIVEN — set up a small local orders database with sample data
import sqlite3

conn = sqlite3.connect("orders.db")
conn.execute("DROP TABLE IF EXISTS orders")
conn.execute("CREATE TABLE orders (order_id TEXT, customer_id TEXT, status TEXT)")
sample_orders = [
    ("ORD-001", "CUST123", "shipped"),
    ("ORD-002", "CUST123", "processing"),
    ("ORD-003", "CUST456", "delivered"),
]
conn.executemany("INSERT INTO orders VALUES (?, ?, ?)", sample_orders)
conn.commit()
conn.close()
print("orders.db ready")


orders.db ready


**Spec for the tool:**
- Signature: `query_orders(customer_id: str) -> dict`
- Connect to `orders.db`, run a **parameterized** query (use `?` placeholders — never
  string-format `customer_id` directly into SQL) selecting `order_id, status` for that customer
- Return `{"orders": [{"order_id": ..., "status": ...}, ...]}`
- If there are no matching rows, return `{"orders": []}` (valid, not an error)


In [94]:
# WRITE THIS: query_orders(customer_id) -> dict, per the spec above

def query_orders(customer_id: str) -> dict:
    conn = sqlite3.connect("orders.db")
    cursor = conn.execute(
        "SELECT order_id, status FROM orders WHERE customer_id = ?",
        (customer_id,)
    )
    rows = cursor.fetchall()
    conn.close()

    orders = [{"order_id": row[0], "status": row[1]} for row in rows]
    return {"orders": orders}


print(query_orders("CUST123"))   # should show 2 orders
print(query_orders("NOBODY"))    # should show an empty list, not crash

{'orders': [{'order_id': 'ORD-001', 'status': 'shipped'}, {'order_id': 'ORD-002', 'status': 'processing'}]}
{'orders': []}


In [95]:
# WRITE THIS: FunctionDeclaration for query_orders, then register all 3 tools together
# (get_weather, search_wikipedia, query_orders) into one Tool/config/tool_registry
query_orders_declaration = types.FunctionDeclaration(
    name="query_orders",
    description="Look up order status and order IDs for a given customer ID in the local orders database.",
    parameters={
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "The customer ID to look up orders for, e.g. 'CUST123'"
            }
        },
        "required": ["customer_id"]
    }
)
all_tools = types.Tool(function_declarations=[
    get_weather_declaration,
    search_wikipedia_declaration,
    query_orders_declaration
])      # replace: types.Tool with all 3 declarations
config = types.GenerateContentConfig(tools = [all_tools])           # replace: GenerateContentConfig
tool_registry = {
    get_weather : get_weather,
    search_wikipedia : search_wikipedia,
    query_orders : query_orders
}       # replace: dict with all 3 functions


### Test full 3-tool chaining


---
## Section 6 — Independent Challenge: Break It, Then Prove It Survives

You already wrapped `get_weather` in resilience via `run_agent_v2` — that wrapping applies
automatically to *every* tool in the registry, including the two you just wrote. Prove it.

**Task:** deliberately trigger a real failure in each tool and confirm the agent still returns
a sensible final answer instead of crashing.


In [96]:
# 🔧 Try to break each tool on purpose, one at a time, and confirm graceful handling:

# 1. A city that doesn't exist
print(run_agent_v2("Weather in Xyzzyxutopia?", config=config, tool_registry=tool_registry, verbose=False))

# 2. A customer ID that doesn't exist
print(run_agent_v2("Order status for customer CUST999?", config=config, tool_registry=tool_registry, verbose=False))

# 3. A Wikipedia query that returns nothing
print(run_agent_v2("Wikipedia fact about qwzxjklplmnbv?", config=config, tool_registry=tool_registry, verbose=False))

I'm sorry, but I couldn't find any weather information for "Xyzzyxutopia". It may not be a recognized city.
I'm sorry, but I encountered a technical error while trying to look up the orders for customer CUST999. Please try again later or contact customer support for assistance.
I couldn't find any Wikipedia pages or information related to "qwzxjklplmnbv", as it appears to be a random string of letters rather than a recognized topic. If you had a different topic in mind, please let me know and I'd be happy to search for that!


---
## Section 7 — Iteration Log Analysis
This is today's core lab task from the slides: **reading a trace and diagnosing a failure.**

First, generate your own real broken trace:


In [97]:
# GIVEN — run with verbose=True so every step prints, then deliberately confuse it
trace_answer = run_agent_v2(
    "What's the weather in Lahoreee and check order status for CUST999?",  # typo'd city + fake customer
    config=config, tool_registry=tool_registry, verbose=True
)
print()
print("FINAL ANSWER:", trace_answer)


[Step 1] action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'error': "unexpected error in get_weather: 'get_weather'"}
[Step 1] action: query_orders({'customer_id': 'CUST999'})
[Step 1] Observation: {'error': "unexpected error in query_orders: 'query_orders'"}
[Step 2] action: get_weather({'city': 'Lahore'})
[Step 2] Observation: {'error': "unexpected error in get_weather: 'get_weather'"}
[Step 2] action: query_orders({'customer_id': 'CUST999'})
[Step 2] Observation: {'error': "unexpected error in query_orders: 'query_orders'"}
[Step 3] Thought: I have enough info. Giving final answer.

FINAL ANSWER: I apologize, but I am currently experiencing technical difficulties retrieving the weather for Lahore and checking the order status for customer CUST999. Please try again later.


Now analyze this **pre-recorded** trace from another (deliberately broken) run — a teammate's
agent that got stuck:

```
[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 34, 'windspeed': 8.2}
[Step 2] Action: query_orders({'customer_id': 'CUST-123'})
[Step 2] Observation: {'orders': []}
[Step 3] Action: query_orders({'customer_id': 'CUST-123'})
[Step 3] Observation: {'orders': []}
[Step 4] Action: query_orders({'customer_id': 'CUST-123'})
[Step 4] Observation: {'orders': []}
Reached max steps without a final answer.
```

### Answer these in the markdown cell below (double-click to edit)
1. The weather lookup worked fine. What went wrong with the orders lookup, specifically?
   (Hint: compare `'CUST-123'` here against the sample data format from Section 5.)
2. Why did the agent call `query_orders` three times with the *exact same* (wrong) argument
   instead of trying something different or giving up sooner?
3. Whose fault is this, really — the model's, the tool's, or the data's? What's the one-line
   fix?


 # **Your analysis:**

1.
2.
3.




Yahan seedha markdown cell mein paste karne layak, concise answers hain:

**Your analysis:**

1. `query_orders` ko `'CUST-123'` (hyphen ke saath) diya gaya, jabke Section 5 ka sample data `"CUST123"` format mein hai (bina hyphen). Exact-match query hone ki wajah se koi row match nahi hui, isliye `{"orders": []}` aaya — ye crash nahi, sirf format mismatch tha.

2. `{"orders": []}` ek valid, error-free response hai — isme koi error signal nahi ke ID ka format ghalat tha. Model ko lagta hai request "successfully" chali bas result empty aaya, isliye wo wahi ID dobara try karta rehta hai, jab tak `max_steps` khatam nahi ho jate.

3. Ye **tool design** ki galti hai, model ki nahi. Tool ne format-mismatch aur "genuinely no orders" mein farq nahi bataya. **One-line fix:** `query_orders` ke description/schema mein exact ID format specify karo (e.g. "IDs look like `CUST123`, no hyphens"), ya tool ke andar customer_id ko normalize (hyphens/spaces strip) karo before querying.